# Notebook d'Inférence - Prédictions Consommation Électrique

Ce notebook permet de faire des prédictions de consommation électrique pour la journée suivante.

## Fonctionnalités :
1. Chargement du modèle depuis MLflow
2. Génération des features pour la journée suivante (toutes les 30 minutes)
3. Prédictions sur toutes les plages horaires
4. Stockage des prédictions en base de données

## Structure de la table de stockage :
- `prediction_id`: TEXT (UUID)
- `prediction_timestamp`: TIMESTAMP
- `prediction_date`: TIMESTAMP
- `prediction_index`: INTEGER
- `prediction`: DOUBLE PRECISION
- `confidence`: DOUBLE PRECISION
- `model_version`: TEXT
- `created_at`: TIMESTAMP

## 0. Initialisation et Configuration

In [1]:
import os
import logging
import warnings
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

✅ Initialisation terminée


## 1. Configuration de la Base de Données

In [2]:
# Configuration PostgreSQL (à adapter selon votre environnement)
DB_URI = os.getenv("PREDICTIONS_POSTGRES_URI")

PREDICTIONS_TABLE = os.getenv("PREDICTIONS_TABLE", "predictions")

if DB_URI:
    print("📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI")
    print(f"  Table: {PREDICTIONS_TABLE}")
else:
    DB_CONFIG = {
        "host": os.getenv("DB_HOST", "localhost"),
        "port": os.getenv("DB_PORT", "5432"),
        "database": os.getenv("DB_NAME", "jinsudai"),
        "user": os.getenv("DB_USER", "postgres"),
        "password": os.getenv("DB_PASSWORD", ""),
    }
    print(f"📊 Configuration DB :")
    print(f"  Host: {DB_CONFIG['host']}")
    print(f"  Port: {DB_CONFIG['port']}")
    print(f"  Database: {DB_CONFIG['database']}")
    print(f"  Table: {PREDICTIONS_TABLE}")

📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI
  Table: predictions


## 2. Configuration MLflow

In [ ]:
## 2. Configuration MLflow
import sys
sys.path.insert(0, str(Path.cwd() / "src"))

from ml.pipelines import InferencePipeline

# Configuration MLflow
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://jinsudai-mlflow.hf.space/")
MODEL_NAME = os.getenv("MODEL_NAME", "consumption_model_test")
MODEL_ALIAS = os.getenv("MODEL_ALIAS", "prod")  # 'prod' ou 'staging'

pipeline = InferencePipeline(
    mlflow_uri=MLFLOW_TRACKING_URI,
    experiment_name=None,
    db_uri=DB_URI
)

pipeline_ready = pipeline.setup()

print("🔧 Pipeline d'inférence initialisé")
print(f"  Modèle: {MODEL_NAME}")
print(f"  Alias: {MODEL_ALIAS}")
print(f"  Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"  Pipeline prêt: {pipeline_ready}")

c:\Users\SustCoop\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\ml-6PV5fMfH-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-15 09:39:46,671 - ml.utils.pipelines.Prediction_pipeline - INFO - Pipeline d'inférence initialisé
2026-06-15 09:39:46,672 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 1: CONFIGURATION ===
2026-06-15 09:39:53,869 - ml.utils.models.models_inference - INFO - MLflow connecté à https://jinsudai-mlflow.hf.space/
2026-06-15 09:39:53,871 - ml.utils.pipelines.Prediction_pipeline - INFO - MLflow configuré
2026-06-15 09:39:54,233 - ml.utils.pipelines.database_handler - INFO - Connexion à la base de données vérifiée
2026-06-15 09:39:54,489 - ml.utils.pipelines.database_handler - INFO - Table predictions_

🔧 Pipeline d'inférence initialisé
  Modèle: consumption_model_test
  Alias: prod
  Tracking URI: https://jinsudai-mlflow.hf.space/
  Pipeline prêt: True


## 3. Chargement du Modèle depuis MLflow

In [4]:
print(f"🔄 Chargement du modèle {MODEL_NAME} (alias: {MODEL_ALIAS})...")

try:
    if not pipeline.load_model(MODEL_NAME, alias_prod=MODEL_ALIAS):
        raise RuntimeError("Échec du chargement du modèle via le pipeline")

    model_info = pipeline.get_model_info() or {}
    model_version = model_info.get("version", "unknown")

    print(f"✅ Modèle chargé avec succès")
    print(f"  Version: {model_version}")

except Exception as e:
    logger.error(f"Erreur lors du chargement du modèle: {e}")
    raise

2026-06-15 09:39:54,534 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 2: CHARGEMENT DU MODÈLE ===


🔄 Chargement du modèle consumption_model_test (alias: prod)...


2026-06-15 09:39:58,246 - ml.utils.models.models_inference - INFO - Model registry source: runs:/e5c4f5a023914c958cdc3829bcf64e15/model
2026-06-15 09:40:07,704 - botocore.credentials - INFO - Found credentials in environment variables.
2026/06/15 09:40:13 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at s3://mlflow-artifacts/2/e5c4f5a023914c958cdc3829bcf64e15/artifacts/model. Returning destination path.
2026-06-15 09:40:42,186 - ml.utils.models.models_inference - WARNING - Chargement sklearn échoué (Could not find a registered artifact repository for: c:\Users\SustCoop\AppData\Local\Temp\tmpff0chg9r. Currently registered schemes are: ['', 'file', 's3', 'r2', 'b2', 'gs', 'wasbs', 'ftp', 'sftp', 'dbfs', 'hdfs', 'viewfs', 'runs', 'models', 'http', 'https', 'mlflow-artifacts', 'abfss']), tentative chargement pyfunc pour models:/consumption_model_test@prod
2026/06/15 09:40:45 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at s3://mlflow-a

✅ Modèle chargé avec succès
  Version: 13


## 4. Génération des Features pour la Journée Suivante

In [5]:

features_df = pipeline.generate_data()


2026-06-15 09:42:01,547 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 3: GÉNÉRATION DES DONNÉES ===
2026-06-15 09:42:09,534 - ml.connectors.holidays.holidays_api - INFO - Données vacances scolaires chargées: 570 périodes
2026-06-15 09:42:09,559 - ml.connectors.holidays.holidays_api - INFO - Vacances scolaires pour [2026] (zone C): 6 périodes
2026-06-15 09:42:12,032 - ml.connectors.holidays.holidays_api - INFO - Jours fériés pour 2026: 11 jours
C:\_sources\Jenedai\jinsudai\src\ml\connectors\holidays\holidays_api.py:477: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["jour férié"] = df["jour férié"].fillna(0).astype(int)


[Aix en Provence] Récupération prévisions pour 1 jour(s)...


2026-06-15 09:42:14,861 - ml.utils.data.data_prediction - INFO - Données d'inférence générées: 48 échantillons
2026-06-15 09:42:14,876 - ml.utils.pipelines.Prediction_pipeline - INFO - df_inference généré avec succès:              Horodate  temperature_2m_mean  relative_humidity_mean  \
0 2026-06-15 00:00:00                 19.8                    66.0   
1 2026-06-15 00:30:00                 20.0                    50.0   
2 2026-06-15 01:00:00                 17.9                    73.0   
3 2026-06-15 01:30:00                 20.0                    50.0   
4 2026-06-15 02:00:00                 17.4                    86.0   

   precipitation_sum  is_vacances jour de la semaine  jour férié  
0                0.0            0             Monday           0  
1                0.0            0             Monday           0  
2                0.0            0             Monday           0  
3                0.0            0             Monday           0  
4                0.0      

OK 24 enregistrements récupérés


## 5. Préparation des Features pour le Modèle

In [6]:

# Préparer les features via PredictionPipeline
features_prep = pipeline.prepare_features()

print(f"✅ Features préparées via PredictionPipeline: {features_prep.shape}")
display(features_prep.head())

2026-06-15 09:42:14,905 - ml.utils.data.data_transformer - INFO - Aucune colonne valide à supprimer après vérification
2026-06-15 09:42:14,912 - ml.utils.data.data_transformer - INFO - Nettoyage des données terminé


✅ Features préparées via PredictionPipeline: (48, 7)


,Horodate,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,jour de la semaine,jour férié
0,2026-06-15 00:00:00,19.8,66.0,0.0,0,Monday,0
1,2026-06-15 00:30:00,20.0,50.0,0.0,0,Monday,0
2,2026-06-15 01:00:00,17.9,73.0,0.0,0,Monday,0
3,2026-06-15 01:30:00,20.0,50.0,0.0,0,Monday,0
4,2026-06-15 02:00:00,17.4,86.0,0.0,0,Monday,0


## 6. Prédictions

In [7]:
print("🔄 Prédictions en cours...")

try:
    if not pipeline_ready:
        raise RuntimeError("Le pipeline d'inférence n'est pas prêt")

    prediction_ok = pipeline.run_predictions()
    if not prediction_ok:
        raise RuntimeError("Échec des prédictions via le pipeline")

    predictions_df = pipeline.get_predictions_df()
    if predictions_df is None or predictions_df.empty:
        raise RuntimeError("Aucune prédiction générée")

    print(f"✅ Prédictions terminées: {len(predictions_df)} valeurs")
    print()
    print("📊 Statistiques des prédictions:")
    print(f"  Min: {predictions_df['prediction'].min():.2f}")
    print(f"  Max: {predictions_df['prediction'].max():.2f}")
    print(f"  Mean: {predictions_df['prediction'].mean():.2f}")
    print(f"  Std: {predictions_df['prediction'].std():.2f}")

    if "confidence" in predictions_df.columns:
        print()
        print(f"📈 Scores de confiance disponibles: {predictions_df['confidence'].shape}")

    display(predictions_df.head())
except Exception as e:
    logger.error(f"Erreur lors des prédictions: {e}")
    raise


2026-06-15 09:42:14,986 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 4: EXÉCUTION DES PRÉDICTIONS ===


🔄 Prédictions en cours...


2026-06-15 09:42:15,960 - ml.utils.models.models_inference - INFO - Prédictions générées pour 48 échantillons
2026-06-15 09:42:15,962 - ml.utils.data.data_prediction - INFO - Prédictions ajoutées au DataFrame: (48, 8)
2026-06-15 09:42:15,964 - ml.utils.pipelines.Prediction_pipeline - INFO - Prédictions générées: 48 échantillons


✅ Prédictions terminées: 48 valeurs

📊 Statistiques des prédictions:
  Min: 556.85
  Max: 613.10
  Mean: 586.07
  Std: 12.03


,Horodate,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,jour de la semaine,jour férié,prediction
0,2026-06-15 00:00:00,19.8,66.0,0.0,0,Monday,0,578.216614
1,2026-06-15 00:30:00,20.0,50.0,0.0,0,Monday,0,583.072998
2,2026-06-15 01:00:00,17.9,73.0,0.0,0,Monday,0,613.100525
3,2026-06-15 01:30:00,20.0,50.0,0.0,0,Monday,0,583.072998
4,2026-06-15 02:00:00,17.4,86.0,0.0,0,Monday,0,583.867188


## 7. Préparation des Données pour Stockage

In [8]:
predictions_df = pipeline.get_predictions_df()

if predictions_df is None or predictions_df.empty:
    raise RuntimeError("Aucune prédiction disponible pour le stockage")

print(f"✅ Données prêtes pour stockage: {len(predictions_df)} enregistrements")
display(predictions_df.head())

✅ Données prêtes pour stockage: 48 enregistrements


,Horodate,temperature_2m_mean,relative_humidity_mean,precipitation_sum,is_vacances,jour de la semaine,jour férié,prediction
0,2026-06-15 00:00:00,19.8,66.0,0.0,0,Monday,0,578.216614
1,2026-06-15 00:30:00,20.0,50.0,0.0,0,Monday,0,583.072998
2,2026-06-15 01:00:00,17.9,73.0,0.0,0,Monday,0,613.100525
3,2026-06-15 01:30:00,20.0,50.0,0.0,0,Monday,0,583.072998
4,2026-06-15 02:00:00,17.4,86.0,0.0,0,Monday,0,583.867188


## 8. Connexion à la Base de Données

In [9]:
if DB_URI:
    print("📌 Le pipeline stockera les prédictions dans la base de données via PredictionPipeline.")
else:
    print("⚠️ Aucune URI DB fournie : le stockage via PredictionPipeline sera ignoré.")

📌 Le pipeline stockera les prédictions dans la base de données via PredictionPipeline.


In [11]:
print("🔄 Stockage des prédictions en cours via PredictionPipeline...")

try:
    if not pipeline_ready:
        raise RuntimeError("Le pipeline d'inférence n'est pas prêt")

    stored = pipeline.store_predictions()
    if not stored:
        raise RuntimeError("Échec du stockage des prédictions via le pipeline")

    print("✅ Prédictions stockées via PredictionPipeline")
except Exception as e:
    logger.error(f"Erreur lors du stockage des prédictions: {e}")
    raise


2026-06-15 09:42:16,135 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 5: STOCKAGE EN BASE DE DONNÉES ===


🔄 Stockage des prédictions en cours via PredictionPipeline...


2026-06-15 09:42:16,162 - ml.utils.pipelines.Prediction_pipeline - INFO - Stockage des prédictions               Horodate  temperature_2m_mean  relative_humidity_mean  precipitation_sum  is_vacances jour de la semaine  jour férié  prediction prediction_timestamp     prediction_date  prediction_index
47 2026-06-15 23:30:00                 20.0                    50.0                0.0            0             Monday           0  583.072998  2026-06-15 23:30:00 2026-06-15 23:30:00                48
46 2026-06-15 23:00:00                 20.1                    82.0                0.0            0             Monday           0  565.838135  2026-06-15 23:00:00 2026-06-15 23:00:00                47
45 2026-06-15 22:30:00                 20.0                    50.0                0.0            0             Monday           0  583.072998  2026-06-15 22:30:00 2026-06-15 22:30:00                46
44 2026-06-15 22:00:00                 21.3                    75.0                0.0       

✅ Prédictions stockées via PredictionPipeline
